In [1]:
# 导入必要的库
import os
import glob
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import random
import csv

In [ ]:
# 检查CUDA是否可用
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Main_path = 'D:\\Desktop\\Desktop\\Python\\Forward_and_Inverse_problem\\'
Data_path = f'{Main_path}Data\\'
save_path = f'{Main_path}Model\\Python_code_FFT_Complex\\'
random.seed(42)

In [3]:
M = 9900
batch_size = 1
label_dim = 3

criterion = nn.MSELoss().to(device)

# 定义数据集类
class SignalDataset(Dataset):
    def __init__(self, excitation_signals, labels, experimental_signal):
        self.excitation_signals = torch.tensor(excitation_signals, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.float32)
        self.experimental_signal = torch.tensor(experimental_signal, dtype=torch.float32)

    def __len__(self):
        return len(self.excitation_signals)

    def __getitem__(self, idx):
        return {
            'excitation_signal': self.excitation_signals[idx],
            'labels': self.labels[idx],
            'experimental_signal': self.experimental_signal[idx]
        }

In [4]:
class ConvAutoencoder(nn.Module):
    def __init__(self, signal_length, label_dim, hidden_dim, dropout):
        super().__init__()
        self.signal_length = signal_length
        self.label_dim = label_dim
        
        # 标签投影网络（7层）
        self.label_projection = nn.Sequential(
            nn.Linear(label_dim, hidden_dim * 4),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 4, hidden_dim * 8),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 8, hidden_dim * 16),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 16, hidden_dim * 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 32, hidden_dim * 64),  # 第5层
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 64, hidden_dim * 128), # 第6层
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 128, hidden_dim * 64) # 第7层
        )
        
        # 7层编码器
        self.encoder = nn.Sequential(
            # Layer 1
            nn.Conv1d(1, hidden_dim, kernel_size=7, stride=2, padding=3),
            nn.ReLU(),
            
            # Layer 2
            nn.Conv1d(hidden_dim, hidden_dim*2, kernel_size=5, stride=2, padding=2),
            nn.ReLU(),
            
            # Layer 3
            nn.Conv1d(hidden_dim*2, hidden_dim*4, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            
            # Layer 4
            nn.Conv1d(hidden_dim*4, hidden_dim*8, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            
            # Layer 5 - 修改
            nn.Conv1d(hidden_dim*8, hidden_dim*16, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            
            # Layer 6 - 修改
            nn.Conv1d(hidden_dim*16, hidden_dim*32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            
            # Layer 7 - 修改
            nn.Conv1d(hidden_dim*32, hidden_dim*64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
        )
        
        # 7层解码器
        self.decoder = nn.Sequential(
            # Layer 1: 融合
            nn.ConvTranspose1d(hidden_dim*64 + hidden_dim*64, hidden_dim*32,  # 标签特征改为64维
                              kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            
            # Layer 2
            nn.ConvTranspose1d(hidden_dim*32, hidden_dim*16, 
                              kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            
            # Layer 3
            nn.ConvTranspose1d(hidden_dim*16, hidden_dim*8, 
                              kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            
            # Layer 4
            nn.ConvTranspose1d(hidden_dim*8, hidden_dim*4, 
                              kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            
            # Layer 5
            nn.ConvTranspose1d(hidden_dim*4, hidden_dim*2, 
                              kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            
            # Layer 6
            nn.ConvTranspose1d(hidden_dim*2, hidden_dim, 
                              kernel_size=5, stride=2, padding=2, output_padding=1),
            nn.ReLU(),
            
            # Layer 7
            nn.ConvTranspose1d(hidden_dim, 1, 
                              kernel_size=7, stride=2, padding=3, output_padding=1),
        )
        
        # 计算编码输出长度
        with torch.no_grad():
            test_input = torch.randn(1, 1, signal_length)
            encoded = self.encoder(test_input)
            self.encoder_output_len = encoded.shape[-1]

    def forward(self, x, labels):
        # 编码
        encoded = self.encoder(x)
        
        # 标签特征
        label_features = self.label_projection(labels)
        label_features = label_features.unsqueeze(-1).expand(-1, -1, self.encoder_output_len)
        
        # 融合和解码
        combined = torch.cat([encoded, label_features], dim=1)
        decoded = self.decoder(combined)
        
        return decoded[:,:,:self.signal_length]

In [5]:
def load_data(excitation_signal_path, experimental_signal_path, M):
    excitation_signals = []  # 存储生成信号
    labels = []  # 存储标签 (Force_label, HI_label, Distance_label, Time_label)
    experimental_signals = []  # 存储输出参数
    filenames = []  # 存储文件名

    # 加载生成信号数据
    excitation_dataPaths = sorted(glob.glob(os.path.join(excitation_signal_path, '**', '*.csv'), recursive=True))
    for dataPath in excitation_dataPaths:
        filename = os.path.basename(dataPath)
        parts = filename.split('_')

        Temperature_label = float(parts[0])
        Time_label = float(parts[1])
        Distance_label = float(parts[4])
        amp_part = parts[5].replace('.csv', '')
        Amp_label = float(amp_part)    

        data_1 = pd.read_csv(dataPath)
        # 找到第一个非0值的位置
        for N in range(len(data_1)):
            if data_1.iloc[N, 1] != 0:
                break
        # 截取长度为M的信号段
        if N + M <= len(data_1):  # 避免越界
            excitation_signal = data_1.iloc[N:N+M, 1].values*Amp_label
        else:
            continue

        # 分别存储 excitation_signal 和 labels
        excitation_signals.append(excitation_signal)
        labels.append([Temperature_label, Time_label, Distance_label])

    # 加载实验信号数据
    experimental_dataPaths = sorted(glob.glob(os.path.join(experimental_signal_path, '**', '*.csv'), recursive=True))
    for dataPath in experimental_dataPaths:
        data_1 = pd.read_csv(dataPath)
        # 找到第一个非0值的位置
        for N in range(len(data_1)):
            if data_1.iloc[N, 1] != 0:
                break
        # 截取长度为M的信号段（避免越界）
        if N + M <= len(data_1):
            experimental_signal = data_1.iloc[N:N+M, 1].values
            experimental_signals.append(experimental_signal)
        else:
            # 如果长度不足，跳过该样本（也可根据需求补0）
            continue

    # 确保 excitation_signals、labels 和 experimental_signals 的长度一致
    if len(excitation_signals) != len(experimental_signals) or len(labels) != len(experimental_signals):
        print(f"Error: 输入输出数量不匹配！生成信号数:{len(excitation_signals)}, 实验信号数:{len(experimental_signals)}, 标签数:{len(labels)}")
        return None, None, None, None, None

    # 转换为 NumPy 数组
    excitation_signals = np.array(excitation_signals, dtype=float)
    labels = np.array(labels, dtype=float)
    experimental_signals = np.array(experimental_signals, dtype=float)

    return excitation_signals, labels, experimental_signals

In [ ]:
excitation_signal_path = f'{Data_path}Hanning_Excitation_signal'
experimental_signal_path = f'{Data_path}Experiment_signal'

# 加载数据
filenames = os.listdir(excitation_signal_path)


# 加载数据（假设 load_data 按文件名顺序返回数据）
excitation_signals, labels, experimental_signal = load_data(
    excitation_signal_path, experimental_signal_path,M)

excitation_signals = excitation_signals
labels = labels
experimental_signal = experimental_signal

# 所有数据作为测试集
test_filenames = filenames

# 所有数据分配给测试集
excitation_signals_test = excitation_signals
labels_test = labels
experimental_signal_test = experimental_signal

# 增加一个维度，从 [batch_size, N] 变为 [batch_size, 1, N]
excitation_signals_test = np.expand_dims(excitation_signals_test, axis=1)
experimental_signal_test = np.expand_dims(experimental_signal_test, axis=1)

print(f'excitation_signals_test:{excitation_signals_test.shape}')
print(f'Total test samples: {len(excitation_signals_test)}')

# 只创建测试集数据集和数据加载器
test_dataset = SignalDataset(excitation_signals_test, labels_test, experimental_signal_test)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 训练集和验证集设为 None（或空列表），避免后续代码报错
train_loader = None
val_loader = None

In [ ]:
excitation_signals_test_1 = excitation_signals_test.reshape(-1,M,1)
plt.style.use('default')
plt.figure(figsize=(10,6))
plt.rcParams['font.family'] = ['Times New Roman']
plt.plot(excitation_signals_test_1[1],linewidth=1.5)
plt.xlabel('Sample point',fontdict={'weight': 'normal', 'size': 18})
plt.ylabel('Amplitude(v)',fontdict={'weight': 'normal', 'size': 18})
#坐标轴刻度大小设置
plt.tick_params(axis='both', which='major', labelsize=15)
plt.xlim([0,M])
plt.savefig(f'{save_path}Forward_problem\\excitation_signal_test.jpg', dpi=600, bbox_inches='tight')

In [ ]:
experimental_signal_test_1 = experimental_signal_test.reshape(-1,M,1)
plt.style.use('default')
plt.figure(figsize=(10,6))
plt.rcParams['font.family'] = ['Times New Roman']
plt.plot(experimental_signal_test_1[25],linewidth=1.5)
plt.xlabel('Sample point',fontdict={'weight': 'normal', 'size': 18})
plt.ylabel('Amplitude(V)',fontdict={'weight': 'normal', 'size': 18})
#坐标轴刻度大小设置
plt.tick_params(axis='both', which='major', labelsize=15)
plt.xlim([0,M])
plt.savefig(f'{save_path}Forward_problem\\Experiment_signal_test.jpg', dpi=600, bbox_inches='tight')

In [ ]:
model = torch.load(f'{save_path}Forward_problem\\Forward_Model_FFT_Complex.pth', 
                   map_location=device)

In [10]:
def calculate_fft(signal, sampling_rate):
    N = len(signal)  # 信号长度
    fft_values = np.fft.fft(signal)  # 计算FFT
    fft_magnitude = np.abs(fft_values)  # 取模
    fft_magnitude = fft_magnitude[:N // 2]  # 只取一半（正频率部分）
    freq = np.fft.fftfreq(N, d=1 / sampling_rate)[:N // 2]  # 频率轴
    return freq, fft_magnitude

In [11]:
def calculate_pcc(x, y):
    """
    计算两个信号的皮尔逊相关系数(PCC)。

    参数:
        x (torch.Tensor): 第一个信号，假定在 CUDA 上。
        y (torch.Tensor): 第二个信号，假定在 CUDA 上。

    返回:
        float: 皮尔逊相关系数。
    """
    
    # 计算均值
    mean_x = torch.mean(x)
    mean_y = torch.mean(y)
    
    # 计算协方差
    covariance = torch.mean((x - mean_x) * (y - mean_y))
    
    # 计算标准差
    std_x = torch.std(x)
    std_y = torch.std(y)
    
    # 计算皮尔逊相关系数
    if std_x == 0 or std_y == 0:
        raise ValueError("标准差不能为零，这可能导致除以零的错误。")
    
    pcc = covariance / (std_x * std_y)

    pcc_1 = pcc.to(device)

    return pcc_1.item()  # 将结果转换为 Python 标量

In [ ]:
pcc_time = 0.0
pcc_fft = 0.0

Error_fft = 0.0

# 采样率
sampling_rate = 2.5e7

signal_predicted = []
signal_origin=[]

running_loss = 0.0
criterion = nn.MSELoss().to(device)

# 将模型设置为评估模式
model.eval()  
# 进行预测
with torch.no_grad():  # 关闭梯度计算
    for batch in test_loader:
        excitation_signal = batch['excitation_signal'].to(device)
        labels = batch['labels'].to(device)
        experimental_signal = batch['experimental_signal'].to(device)

        experimental_signal_pred = model(excitation_signal, labels)
        loss = criterion(experimental_signal_pred, experimental_signal)

        running_loss += loss.item()
        pcc_time += calculate_pcc(experimental_signal_pred,experimental_signal)

        # 将重构信号移动到CPU并转换为NumPy数组
        Origin_x3 = experimental_signal.cpu().numpy().squeeze()
        predicted_x3 = experimental_signal_pred.cpu().numpy().squeeze()

        # 假设 Original_signal 和 predicted_signal 已经定义
        freq_origin, fft_magnitude_origin = calculate_fft(Origin_x3, sampling_rate)
        freq_predicted, fft_magnitude_predicted = calculate_fft(predicted_x3, sampling_rate)

        fft_magnitude_predicted,fft_magnitude_origin=torch.from_numpy(fft_magnitude_predicted),torch.from_numpy(fft_magnitude_origin),

        Error_fft += criterion(fft_magnitude_predicted,fft_magnitude_origin)
        pcc_fft += calculate_pcc(fft_magnitude_predicted,fft_magnitude_origin)
    
        signal_origin.append(experimental_signal)
        signal_predicted.append(experimental_signal_pred)

    test_loss_time = running_loss / len(test_loader)
    test_pcc_time = pcc_time / len(test_loader)

    test_fft_error = Error_fft / len(test_loader)
    test_fft_pcc = pcc_fft / len(test_loader)

    print(f'Time_error:{test_loss_time:.4f}')
    print(f'Time_PCC:{test_pcc_time:.4f}')
    print(f'FFT_error:{test_fft_error:.4f}')
    print(f'FFT_PCC:{test_fft_pcc:.4f}')

In [ ]:
# 定义要操作的文件夹路径
results_paths = [
    os.path.join(Data_path, "Results_FFT_Complex\\Predicted_signal\\"),
    os.path.join(Data_path, "Results_FFT_Complex\\Predicted_FFT\\"),
    os.path.join(Data_path, "Results_FFT_Complex\\Time_Difference\\"),
    os.path.join(Data_path, "Results_FFT_Complex\\Difference_FFT\\"),
    os.path.join(save_path, "Forward_problem\\")  # 也确保这个文件夹存在
]

# 首先确保所有需要的文件夹都存在
for folder_path in results_paths:
    # 确保路径分隔符正确（可选，根据系统调整）
    folder_path = os.path.normpath(folder_path)
    
    # 如果文件夹不存在，创建它
    if not os.path.exists(folder_path):
        try:
            os.makedirs(folder_path, exist_ok=True)
            print(f"Created directory: {folder_path}")
        except Exception as e:
            print(f"Failed to create directory {folder_path}. Reason: {e}")

# 现在处理删除操作
results_1 = os.path.join(Data_path, "Results_FFT_Complex\\Predicted_signal\\")
if os.path.exists(results_1):
    for filename in os.listdir(results_1):
        file_path = os.path.join(results_1, filename)
        try:
            if os.path.isfile(file_path):
                os.remove(file_path)
        except Exception as e:
            print(f"Failed to delete {file_path}. Reason: {e}")
else:
    print(f"The directory {results_1} does not exist. It has been created.")

results_2 = os.path.join(Data_path, "Results_FFT_Complex\\Predicted_FFT\\")
if os.path.exists(results_2):
    for filename in os.listdir(results_2):
        file_path = os.path.join(results_2, filename)
        try:
            if os.path.isfile(file_path):
                os.remove(file_path)
        except Exception as e:
            print(f"Failed to delete {file_path}. Reason: {e}")
else:
    print(f"The directory {results_2} does not exist. It has been created.")

results_3 = os.path.join(Data_path, "Results_FFT_Complex\\Time_Difference\\")
if os.path.exists(results_3):
    for filename in os.listdir(results_3):
        file_path = os.path.join(results_3, filename)
        try:
            if os.path.isfile(file_path):
                os.remove(file_path)
        except Exception as e:
            print(f"Failed to delete {file_path}. Reason: {e}")
else:
    print(f"The directory {results_3} does not exist. It has been created.")

results_4 = os.path.join(Data_path, "Results_FFT_Complex\\Difference_FFT\\")
if os.path.exists(results_4):
    for filename in os.listdir(results_4):
        file_path = os.path.join(results_4, filename)
        try:
            if os.path.isfile(file_path):
                os.remove(file_path)
        except Exception as e:
            print(f"Failed to delete {file_path}. Reason: {e}")
else:
    print(f"The directory {results_4} does not exist. It has been created.")

# 定义文件名
file_name = f'Test_Error_FFT_Complex.csv'
file_path = os.path.join(save_path, "Forward_problem", file_name)

# 确保父文件夹存在
os.makedirs(os.path.dirname(file_path), exist_ok=True)

# 检查文件是否存在，如果存在则删除
if os.path.exists(file_path):
    os.remove(file_path)
    print(f"Deleted existing file: {file_path}")

In [14]:
# 寻找指定频率范围内的最大幅值
def find_max_in_range(freq, magnitude, freq_range):
    mask = (freq / 1e3 >= freq_range[0]) & (freq / 1e3 <= freq_range[1])
    max_index = np.argmax(magnitude[mask])
    max_freq = freq[mask][max_index] / 1e3
    max_magnitude = magnitude[mask][max_index]
    return max_freq, max_magnitude

In [ ]:
# 遍历每个信号
for a in range(len(experimental_signal_test)):
    # 获取对应的 CSV 文件名
    csv_filename = test_filenames[a]
    # 去掉 CSV 文件名的扩展名，用于保存图片
    csv_filename_without_extension = os.path.splitext(csv_filename)[0]

    # 将重构信号移动到CPU并转换为NumPy数
    signal_origin_3 = signal_origin[a].cpu().numpy().squeeze()
    predicted_signal = signal_predicted[a].cpu().numpy().squeeze()

    # 计算信号的差值
    difference = signal_origin_3 - predicted_signal
    Time_RMSE = np.sqrt(np.mean((signal_origin_3 - predicted_signal) ** 2))

    # 采样率
    sampling_rate = 2.5e7

    # 假设 Original_signal 和 predicted_signal 已经定义
    freq_origin, fft_magnitude_origin = calculate_fft(signal_origin_3, sampling_rate)
    freq_predicted, fft_magnitude_predicted = calculate_fft(predicted_signal, sampling_rate)
    freq_difference, fft_magnitude_difference = calculate_fft(difference, sampling_rate)
    fft_magnitude_predicted, fft_magnitude_origin, fft_magnitude_difference = torch.from_numpy(fft_magnitude_predicted), torch.from_numpy(fft_magnitude_origin),torch.from_numpy(fft_magnitude_difference)

    # 绘制时域信号图
    plt.style.use('default')
    plt.figure(figsize=(10, 6))
    plt.rcParams['font.family'] = ['Times New Roman']
    plt.plot(signal_origin_3, linewidth=1.5, label='Experimental Signal')
    plt.plot(predicted_signal, linewidth=1.5, linestyle='--', label='Predicted Signal')

    plt.xlabel('Sample point', fontdict={'weight': 'normal', 'size': 18})
    plt.ylabel('Amplitude(V)', fontdict={'weight': 'normal', 'size': 18})
    plt.tick_params(axis='both', which='major', labelsize=15)
    plt.legend(loc='upper right', fontsize=20)
    plt.text(max(signal_origin_3), max(signal_origin_3), f'RMSE = {Time_RMSE:.4f}', fontsize=28, verticalalignment='top')
    plt.savefig(f'{Data_path}Results_FFT_Complex\\Predicted_signal\\{csv_filename_without_extension}.jpg', dpi=600, bbox_inches='tight')
    plt.close()

    # 绘制差值图
    plt.figure(figsize=(10, 6))
    plt.rcParams['font.family'] = ['Times New Roman']
    plt.plot(difference, linewidth=1.5)
    plt.xlim([0, M])
    plt.ylim([-max(difference), max(difference)])

    plt.xlabel('Sample point', fontdict={'weight': 'normal', 'size': 18})
    plt.ylabel('Difference (V)', fontdict={'weight': 'normal', 'size': 18})
    plt.tick_params(axis='both', which='major', labelsize=15)
    plt.legend(loc='upper right', fontsize=20)
    plt.savefig(f'{Data_path}Results_FFT_Complex\\Time_Difference\\{csv_filename_without_extension}.jpg', dpi=600, bbox_inches='tight')
    plt.close()


    # 原始信号
    A1_freq, Af = find_max_in_range(freq_origin, fft_magnitude_origin, (40, 60))
    As_freq, As = find_max_in_range(freq_origin, fft_magnitude_origin, (80, 120))

    # 预测信号
    Bf_freq, Bf = find_max_in_range(freq_predicted, fft_magnitude_predicted, (40, 60))
    Bs_freq, Bs = find_max_in_range(freq_predicted, fft_magnitude_predicted, (80, 120))

    Af_max = max(Af,Bf)
    As_max = max(As,Bs)

    # 绘制FFT频谱图
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(freq_origin / 1e3, fft_magnitude_origin, label='Experimental Signal')
    ax.plot(freq_predicted / 1e3, fft_magnitude_predicted, linestyle='--', label='Predicted Signal', color='orange')
    ax.set_xlabel('Frequency (kHz)', fontdict={'weight': 'normal', 'size': 18})
    ax.set_ylabel('FFT Magnitude', fontdict={'weight': 'normal', 'size': 18})
    ax.legend()
    ax.set_xlim([10, 120])
    ax.set_ylim([0, Af_max])
    ax.tick_params(axis='both', which='major', labelsize=15)
    ax.legend(loc='upper right', fontsize=20)

    # 创建局部放大图
    axins = inset_axes(ax, width="40%", height="40%", loc='lower left',
                       bbox_to_anchor=(0.55, 0.1, 1, 1), bbox_transform=ax.transAxes, borderpad=1)
    axins.plot(freq_origin / 1e3, fft_magnitude_origin, label='Experimental Signal')
    axins.plot(freq_predicted / 1e3, fft_magnitude_predicted, linestyle='--', label='Predicted Signal', color='orange')
    axins.set_xlim([80, 120])
    axins.set_ylim([0, As_max])
    axins.tick_params(axis='both', which='major', labelsize=10)

    FFT_RMSE = torch.sqrt(torch.mean((fft_magnitude_origin - fft_magnitude_predicted) ** 2))
    
    # 计算NP_true和NP_pre
    NP_true = As / (Af ** 2)
    NP_pre = Bs / (Bf ** 2)

    # 计算误差
    NP_Error = abs(NP_true - NP_pre) / NP_true * 100

    # 在图片中显示这些值
    ax.text(0.05, 0.95, f'RMSE = \n{FFT_RMSE:.4f}\n\nNP_true = \n{NP_true:.8f}\n\nNP_pre = \n{NP_pre:.8f}\n\nError = \n{NP_Error:.4f}%',
            transform=ax.transAxes, fontsize=22, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))

    plt.savefig(f'{Data_path}Results_FFT_Complex\\Predicted_FFT\\{csv_filename_without_extension}.jpg', dpi=600, bbox_inches='tight')
    plt.close()

    print(f"{csv_filename_without_extension}, {a+1}/{len(experimental_signal_test)}, NP_Error: {NP_Error:.4f}%, Time_RMSE = {Time_RMSE:.4f},FFT_RMSE = {FFT_RMSE:.8f}")

    # 绘制FFT频谱图
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(freq_difference / 1e3, fft_magnitude_difference, label='Difference Signal FFT')
    ax.set_xlabel('Frequency (kHz)', fontdict={'weight': 'normal', 'size': 18})
    ax.set_ylabel('FFT Magnitude', fontdict={'weight': 'normal', 'size': 18})
    ax.tick_params(axis='both', which='major', labelsize=15)
    ax.legend(loc='upper right', fontsize=20)
    ax.set_xlim([10, 150])
    plt.savefig(f'{Data_path}Results_FFT_Complex\\Difference_FFT\\{csv_filename_without_extension}.jpg', dpi=600, bbox_inches='tight')
    plt.close()

    # 打开 CSV 文件，如果文件不存在会自动创建
    with open('Test_Error_FFT_Complex.csv', mode='a', newline='') as file:
        writer = csv.writer(file)
        # 写入每一行的数据
        writer.writerow([csv_filename_without_extension, f"{a+1}/{len(experimental_signal_test)}", f"{NP_Error:.4f}", f"{Time_RMSE:.4f}"])